In [ ]:
from pathlib import Path
import zipfile
import hashlib

ZIP_PATH = Path("/content/bob_cyclone_training_v1.zip")
EXTRACT_DIR = Path("/content/bob_cyclone_training_v1")

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

# Calculate the package checksum.
sha256 = hashlib.sha256()
with ZIP_PATH.open("rb") as file:
    for block in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(block)

print("ZIP size:", round(ZIP_PATH.stat().st_size / (1024**2), 2), "MB")
print("SHA-256:", sha256.hexdigest())

# Safely inspect and extract the package.
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()
    assert bad_file is None, f"Corrupted ZIP member: {bad_file}"

    members = archive.infolist()
    print("\nFiles inside ZIP:")

    for member in members:
        size_mb = member.file_size / (1024**2)
        print(f"  {member.filename:<60} {size_mb:>8.2f} MB")

    for member in members:
        destination = (EXTRACT_DIR / member.filename).resolve()
        if EXTRACT_DIR.resolve() not in destination.parents and destination != EXTRACT_DIR.resolve():
            raise ValueError(f"Unsafe ZIP path: {member.filename}")

    archive.extractall(EXTRACT_DIR)

print("\nExtraction completed:", EXTRACT_DIR)
print("Package integrity check passed.")

In [ ]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/bob_cyclone_training_v1")

EXPECTED_ROWS = {
    "train": 4560,
    "validation": 854,
    "test": 1018,
}

def load_partition(name):
    folder = ROOT / name
    parquet_files = sorted(folder.glob("*.parquet"))

    assert parquet_files, f"No Parquet files found in {folder}"

    frames = [pd.read_parquet(path) for path in parquet_files]
    frame = pd.concat(frames, ignore_index=True)

    print(
        f"{name:<12} "
        f"rows={len(frame):>5}  "
        f"columns={len(frame.columns):>3}  "
        f"files={len(parquet_files)}"
    )

    return frame

train_df = load_partition("train")
validation_df = load_partition("validation")
test_df = load_partition("test")

partitions = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

print("\nChecking expected row counts...")

for name, frame in partitions.items():
    actual = len(frame)
    expected = EXPECTED_ROWS[name]

    assert actual == expected, (
        f"{name}: expected {expected} rows, found {actual}"
    )

    print(f"  {name}: {actual} rows — PASS")

reference_columns = list(train_df.columns)

print("\nChecking matching schemas...")

for name, frame in partitions.items():
    assert list(frame.columns) == reference_columns, (
        f"{name} schema does not match training schema"
    )

print(f"  All partitions have the same {len(reference_columns)} columns — PASS")

required_columns = ["SID", "ISO_TIME"]

for column in required_columns:
    assert column in train_df.columns, f"Required column missing: {column}"

print("\nChecking key integrity...")

for name, frame in partitions.items():
    frame["ISO_TIME"] = pd.to_datetime(frame["ISO_TIME"], errors="coerce")

    null_sid = int(frame["SID"].isna().sum())
    invalid_time = int(frame["ISO_TIME"].isna().sum())
    duplicate_keys = int(frame.duplicated(["SID", "ISO_TIME"]).sum())

    assert null_sid == 0, f"{name}: {null_sid} missing SID values"
    assert invalid_time == 0, f"{name}: {invalid_time} invalid timestamps"
    assert duplicate_keys == 0, (
        f"{name}: {duplicate_keys} duplicate SID/ISO_TIME keys"
    )

    print(
        f"  {name}: missing SID={null_sid}, "
        f"invalid time={invalid_time}, "
        f"duplicate keys={duplicate_keys} — PASS"
    )

train_sids = set(train_df["SID"].astype(str))
validation_sids = set(validation_df["SID"].astype(str))
test_sids = set(test_df["SID"].astype(str))

overlap_train_validation = train_sids & validation_sids
overlap_train_test = train_sids & test_sids
overlap_validation_test = validation_sids & test_sids

assert not overlap_train_validation, "Train/validation cyclone leakage detected"
assert not overlap_train_test, "Train/test cyclone leakage detected"
assert not overlap_validation_test, "Validation/test cyclone leakage detected"

print("\nCyclone leakage check — PASS")
print("  Train cyclones:", len(train_sids))
print("  Validation cyclones:", len(validation_sids))
print("  Test cyclones:", len(test_sids))

print("\nTime coverage:")

for name, frame in partitions.items():
    print(
        f"  {name:<12} "
        f"{frame['ISO_TIME'].min()} to {frame['ISO_TIME'].max()}"
    )

print("\nChecking chronological order within cyclones...")

for name, frame in partitions.items():
    ordered = (
        frame.sort_values(["SID", "ISO_TIME"])
        [["SID", "ISO_TIME"]]
        .reset_index(drop=True)
    )

    current = (
        frame[["SID", "ISO_TIME"]]
        .reset_index(drop=True)
    )

    is_ordered = current.equals(ordered)

    print(
        f"  {name}: "
        f"{'already ordered' if is_ordered else 'requires loader sorting'}"
    )

target_candidates = [
    column for column in reference_columns
    if column.upper().startswith("TARGET")
]

print("\nTarget columns:")
for column in target_candidates:
    print(" ", column)

assert len(target_candidates) >= 4, (
    "Expected at least four prediction-target columns"
)

print("\nMissing target values:")

for name, frame in partitions.items():
    missing_targets = int(frame[target_candidates].isna().sum().sum())
    assert missing_targets == 0, (
        f"{name}: {missing_targets} missing target values"
    )
    print(f"  {name}: {missing_targets} — PASS")

with (ROOT / "training_manifest.json").open(
    "r", encoding="utf-8"
) as file:
    manifest = json.load(file)

with (ROOT / "normalization_train_only.json").open(
    "r", encoding="utf-8"
) as file:
    normalization = json.load(file)

print("\nManifest top-level fields:")
print(list(manifest.keys()))

print("\nNormalization top-level fields:")
print(list(normalization.keys()))

print("\nDataset columns:")
print(reference_columns)

print("\n" + "=" * 70)
print("V1 STRUCTURAL ACCEPTANCE TEST PASSED")
print("Total usable samples:", sum(len(df) for df in partitions.values()))
print("=" * 70)

In [ ]:
import pandas as pd

actual_numeric_targets = [
    "TARGET_LAT_6H",
    "TARGET_LON_6H",
    "TARGET_WIND_KTS_6H",
    "TARGET_PRESSURE_HPA_6H",
]

print("TARGET COMPLETENESS")
print("=" * 75)

target_report = []

for split_name, frame in partitions.items():
    for column in actual_numeric_targets:
        missing = int(frame[column].isna().sum())
        available = int(frame[column].notna().sum())
        coverage = available / len(frame) * 100

        target_report.append({
            "split": split_name,
            "target": column,
            "rows": len(frame),
            "available": available,
            "missing": missing,
            "coverage_percent": round(coverage, 2),
        })

target_report_df = pd.DataFrame(target_report)

display(
    target_report_df.pivot(
        index="target",
        columns="split",
        values=["available", "missing", "coverage_percent"],
    )
)

print("\nTRAINING FEATURE MISSINGNESS")
print("=" * 75)

excluded_from_features = {
    "SID",
    "NAME",
    "ISO_TIME",
    "TARGET_TIME_6H",
    "TARGET_LAT_6H",
    "TARGET_LON_6H",
    "TARGET_WIND_KTS_6H",
    "TARGET_PRESSURE_HPA_6H",
}

candidate_features = [
    column for column in train_df.columns
    if column not in excluded_from_features
]

feature_missing = (
    train_df[candidate_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missing = pd.DataFrame({
    "missing": feature_missing,
    "coverage_percent": (
        100 - feature_missing / len(train_df) * 100
    ).round(2),
})

display(feature_missing[feature_missing["missing"] > 0])

print("\nTARGET SUMMARY")
print("=" * 75)

for column in actual_numeric_targets:
    print(f"\n{column}")
    print(
        train_df[column]
        .describe()
        .round(3)
        .to_string()
    )

In [ ]:
from pprint import pprint
import json
import pandas as pd

print("ALL 49 COLUMNS")
print("=" * 80)

for index, column in enumerate(train_df.columns, start=1):
    print(
        f"{index:>2}. "
        f"{column:<32} "
        f"{str(train_df[column].dtype):<20}"
    )

print("\nMANIFEST")
print("=" * 80)
pprint(manifest, sort_dicts=False)

print("\nNORMALIZATION FILE")
print("=" * 80)
pprint(normalization, sort_dicts=False)

print("\nGEOGRAPHIC COVERAGE BY SPLIT")
print("=" * 80)

geographic_report = []

for split_name, frame in partitions.items():
    inside_bay = (
        frame["LAT"].between(5, 25) &
        frame["LON"].between(78, 100)
    )

    east_coast_window = (
        frame["LAT"].between(8, 23) &
        frame["LON"].between(78, 92)
    )

    geographic_report.append({
        "split": split_name,
        "rows": len(frame),
        "min_lat": frame["LAT"].min(),
        "max_lat": frame["LAT"].max(),
        "min_lon": frame["LON"].min(),
        "max_lon": frame["LON"].max(),
        "inside_bay_rows": int(inside_bay.sum()),
        "inside_bay_percent": round(inside_bay.mean() * 100, 2),
        "east_coast_window_rows": int(east_coast_window.sum()),
        "east_coast_window_percent": round(
            east_coast_window.mean() * 100, 2
        ),
    })

display(pd.DataFrame(geographic_report))

print("\nMISSINGNESS INDICATORS REQUIRED")
print("=" * 80)

columns_needing_masks = [
    "WIND_KTS",
    "PRESSURE_HPA",
    "WIND_CHANGE_6H_KTS",
    "PRESSURE_CHANGE_6H_HPA",
    "SST_C",
]

for column in columns_needing_masks:
    if column in train_df.columns:
        print(
            f"{column:<28} "
            f"missing={train_df[column].isna().sum():>4}  "
            f"available={train_df[column].notna().mean() * 100:>6.2f}%"
        )

In [ ]:
from pathlib import Path
from pprint import pprint
import json
import pandas as pd

ROOT = Path("/content/bob_cyclone_training_v1")

with (ROOT / "training_manifest.json").open(
    "r", encoding="utf-8"
) as file:
    manifest = json.load(file)

with (ROOT / "normalization_train_only.json").open(
    "r", encoding="utf-8"
) as file:
    normalization = json.load(file)

print("MANIFEST")
print("=" * 80)
pprint(manifest, sort_dicts=False)

print("\nNORMALIZATION FILE")
print("=" * 80)
pprint(normalization, sort_dicts=False)

print("\nGEOGRAPHIC COVERAGE")
print("=" * 80)

geographic_report = []

for split_name, frame in partitions.items():
    inside_bay = (
        frame["LAT"].between(5, 25) &
        frame["LON"].between(78, 100)
    )

    east_coast_window = (
        frame["LAT"].between(8, 23) &
        frame["LON"].between(78, 92)
    )

    geographic_report.append({
        "split": split_name,
        "rows": len(frame),
        "latitude_range": (
            round(frame["LAT"].min(), 2),
            round(frame["LAT"].max(), 2),
        ),
        "longitude_range": (
            round(frame["LON"].min(), 2),
            round(frame["LON"].max(), 2),
        ),
        "inside_bay_rows": int(inside_bay.sum()),
        "inside_bay_percent": round(
            inside_bay.mean() * 100, 2
        ),
        "east_coast_rows": int(east_coast_window.sum()),
        "east_coast_percent": round(
            east_coast_window.mean() * 100, 2
        ),
    })

display(pd.DataFrame(geographic_report))

print("\nMISSING INPUT FEATURES")
print("=" * 80)

columns_needing_masks = [
    "WIND_KTS",
    "PRESSURE_HPA",
    "SST_C",
    "LAT_CHANGE_6H",
    "LON_CHANGE_6H",
    "WIND_CHANGE_6H_KTS",
    "PRESSURE_CHANGE_6H_HPA",
]

for column in columns_needing_masks:
    missing = int(train_df[column].isna().sum())
    coverage = train_df[column].notna().mean() * 100

    print(
        f"{column:<28} "
        f"missing={missing:>4}  "
        f"coverage={coverage:>6.2f}%"
    )

print("\nJSON files loaded successfully.")

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/bob_cyclone_training_v1")

FEATURE_COLUMNS = manifest["feature_columns"]

TARGET_COLUMNS = [
    "TARGET_LAT_6H",
    "TARGET_LON_6H",
    "TARGET_WIND_KTS_6H",
    "TARGET_PRESSURE_HPA_6H",
]

# Verify the declared schema.
for column in FEATURE_COLUMNS + TARGET_COLUMNS:
    for split_name, frame in partitions.items():
        assert column in frame.columns, (
            f"{column} is missing from {split_name}"
        )

# Add a mask for every predictor that contains missing values
# in at least one partition.
MASKED_FEATURE_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if any(
        frame[column].isna().any()
        for frame in partitions.values()
    )
]

MODEL_FEATURE_COLUMNS = (
    FEATURE_COLUMNS +
    [f"{column}_MISSING" for column in MASKED_FEATURE_COLUMNS]
)

print("Original features:", len(FEATURE_COLUMNS))
print("Missingness masks:", len(MASKED_FEATURE_COLUMNS))
print("Final model features:", len(MODEL_FEATURE_COLUMNS))
print("\nMasked input features:")
for column in MASKED_FEATURE_COLUMNS:
    print(" ", column)

# Target normalization must also come only from training data.
target_normalization = {}

for column in TARGET_COLUMNS:
    valid_values = train_df[column].dropna().astype(float)

    mean = float(valid_values.mean())
    std = float(valid_values.std(ddof=0))

    if not np.isfinite(std) or std < 1e-8:
        std = 1.0

    target_normalization[column] = {
        "mean": mean,
        "std": std,
        "valid_count": int(valid_values.size),
    }

def preprocess_partition(frame):
    processed = pd.DataFrame(index=frame.index)

    # Normalize original features using only the supplied
    # training statistics.
    for column in FEATURE_COLUMNS:
        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        ).astype(float)

        if column in MASKED_FEATURE_COLUMNS:
            processed[f"{column}_MISSING"] = (
                values.isna().astype(np.float32)
            )

        stats = normalization[column]
        mean = float(stats["mean"])
        std = float(stats["std"])

        if not np.isfinite(std) or abs(std) < 1e-8:
            std = 1.0

        # Mean imputation becomes zero after normalization.
        values = values.fillna(mean)
        processed[column] = (
            (values - mean) / std
        ).astype(np.float32)

    # Reorder the columns deterministically.
    processed = processed[MODEL_FEATURE_COLUMNS]

    target_values = np.zeros(
        (len(frame), len(TARGET_COLUMNS)),
        dtype=np.float32,
    )

    target_masks = np.zeros_like(
        target_values,
        dtype=np.float32,
    )

    for target_index, column in enumerate(TARGET_COLUMNS):
        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        ).astype(float)

        valid = values.notna().to_numpy()
        stats = target_normalization[column]

        standardized = (
            (values.fillna(stats["mean"]) - stats["mean"])
            / stats["std"]
        )

        target_values[:, target_index] = (
            standardized.to_numpy(dtype=np.float32)
        )

        target_masks[:, target_index] = (
            valid.astype(np.float32)
        )

    metadata = frame[
        ["SID", "ISO_TIME", "TARGET_TIME_6H", "LAT", "LON"]
    ].copy()

    metadata["INSIDE_BAY"] = (
        frame["LAT"].between(5, 25) &
        frame["LON"].between(78, 100)
    ).astype(np.int8)

    return (
        processed.to_numpy(dtype=np.float32),
        target_values,
        target_masks,
        metadata,
    )

processed = {}

for split_name, frame in partitions.items():
    X, y, y_mask, metadata = preprocess_partition(frame)

    assert np.isfinite(X).all(), (
        f"{split_name}: non-finite model inputs"
    )

    assert np.isfinite(y).all(), (
        f"{split_name}: non-finite standardized targets"
    )

    assert set(np.unique(y_mask)).issubset({0.0, 1.0})

    processed[split_name] = {
        "X": X,
        "y": y,
        "y_mask": y_mask,
        "metadata": metadata,
    }

    valid_targets = y_mask.sum(axis=0).astype(int)

    print(f"\n{split_name.upper()}")
    print("  X shape:", X.shape)
    print("  y shape:", y.shape)
    print(
        "  Valid targets:",
        dict(zip(TARGET_COLUMNS, valid_targets)),
    )
    print(
        "  Bay samples:",
        int(metadata["INSIDE_BAY"].sum()),
    )

preprocessing_contract = {
    "dataset_version": manifest["dataset_version"],
    "model_feature_columns": MODEL_FEATURE_COLUMNS,
    "masked_feature_columns": MASKED_FEATURE_COLUMNS,
    "feature_normalization": normalization,
    "target_columns": TARGET_COLUMNS,
    "target_normalization": target_normalization,
    "missing_input_policy": (
        "Training-mean imputation after adding missingness masks"
    ),
    "missing_target_policy": (
        "Ignore unavailable targets using masked loss"
    ),
    "bay_evaluation_box": {
        "latitude": [5.0, 25.0],
        "longitude": [78.0, 100.0],
    },
}

CONTRACT_PATH = ROOT / "colab_preprocessing_contract.json"

with CONTRACT_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        preprocessing_contract,
        file,
        indent=2,
    )

print("\nPreprocessing contract saved:", CONTRACT_PATH)
print("PREPROCESSING ACCEPTANCE TEST PASSED")

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

def make_loader(split_name, batch_size, shuffle):
    split = processed[split_name]

    dataset = TensorDataset(
        torch.from_numpy(split["X"]),
        torch.from_numpy(split["y"]),
        torch.from_numpy(split["y_mask"]),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

train_loader = make_loader(
    "train",
    batch_size=128,
    shuffle=True,
)

validation_loader = make_loader(
    "validation",
    batch_size=256,
    shuffle=False,
)

class SmokeMultitaskModel(nn.Module):
    def __init__(self, input_size, output_size=4):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, output_size),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = SmokeMultitaskModel(
    input_size=len(MODEL_FEATURE_COLUMNS),
).to(device)

def masked_multitask_mse(
    predictions,
    targets,
    masks,
):
    task_losses = []

    for task_index in range(predictions.shape[1]):
        valid = masks[:, task_index] > 0

        if valid.any():
            task_loss = nn.functional.mse_loss(
                predictions[valid, task_index],
                targets[valid, task_index],
            )

            task_losses.append(task_loss)

    if not task_losses:
        raise RuntimeError(
            "Batch contains no valid prediction targets"
        )

    return torch.stack(task_losses).mean()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("\nModel parameters:", sum(
    parameter.numel()
    for parameter in model.parameters()
))

# -----------------------------
# One training epoch
# -----------------------------

model.train()

training_loss_sum = 0.0
training_rows = 0

for batch_X, batch_y, batch_mask in train_loader:
    batch_X = batch_X.to(device)
    batch_y = batch_y.to(device)
    batch_mask = batch_mask.to(device)

    optimizer.zero_grad(set_to_none=True)

    predictions = model(batch_X)

    loss = masked_multitask_mse(
        predictions,
        batch_y,
        batch_mask,
    )

    assert torch.isfinite(loss), (
        "Non-finite training loss encountered"
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=5.0,
    )

    optimizer.step()

    batch_rows = batch_X.shape[0]
    training_loss_sum += loss.item() * batch_rows
    training_rows += batch_rows

training_loss = training_loss_sum / training_rows

print(f"\nTraining loss: {training_loss:.6f}")

# -----------------------------
# Validation predictions
# -----------------------------

model.eval()

prediction_batches = []
target_batches = []
mask_batches = []

with torch.no_grad():
    for batch_X, batch_y, batch_mask in validation_loader:
        batch_X = batch_X.to(device)

        predictions = model(batch_X)

        prediction_batches.append(
            predictions.cpu().numpy()
        )

        target_batches.append(
            batch_y.numpy()
        )

        mask_batches.append(
            batch_mask.numpy()
        )

validation_predictions_scaled = np.concatenate(
    prediction_batches,
    axis=0,
)

validation_targets_scaled = np.concatenate(
    target_batches,
    axis=0,
)

validation_masks = np.concatenate(
    mask_batches,
    axis=0,
)

assert np.isfinite(
    validation_predictions_scaled
).all()

# Convert predictions and targets back to physical units.
validation_predictions = np.zeros_like(
    validation_predictions_scaled,
)

validation_targets = np.zeros_like(
    validation_targets_scaled,
)

for target_index, column in enumerate(TARGET_COLUMNS):
    mean = target_normalization[column]["mean"]
    std = target_normalization[column]["std"]

    validation_predictions[:, target_index] = (
        validation_predictions_scaled[:, target_index]
        * std
        + mean
    )

    validation_targets[:, target_index] = (
        validation_targets_scaled[:, target_index]
        * std
        + mean
    )

def masked_mae(
    predictions,
    targets,
    mask,
):
    valid = mask > 0

    if not valid.any():
        return float("nan")

    return float(
        np.mean(
            np.abs(
                predictions[valid] -
                targets[valid]
            )
        )
    )

def haversine_km(
    true_lat,
    true_lon,
    predicted_lat,
    predicted_lon,
):
    earth_radius_km = 6371.0088

    true_lat = np.radians(true_lat)
    true_lon = np.radians(true_lon)
    predicted_lat = np.radians(predicted_lat)
    predicted_lon = np.radians(predicted_lon)

    delta_lat = predicted_lat - true_lat
    delta_lon = predicted_lon - true_lon

    a = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(true_lat)
        * np.cos(predicted_lat)
        * np.sin(delta_lon / 2.0) ** 2
    )

    return (
        2.0
        * earth_radius_km
        * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )

validation_metrics = {}

for target_index, column in enumerate(TARGET_COLUMNS):
    validation_metrics[f"{column}_MAE"] = masked_mae(
        validation_predictions[:, target_index],
        validation_targets[:, target_index],
        validation_masks[:, target_index],
    )

track_error_km = haversine_km(
    validation_targets[:, 0],
    validation_targets[:, 1],
    validation_predictions[:, 0],
    validation_predictions[:, 1],
)

bay_mask = (
    processed["validation"]["metadata"]["INSIDE_BAY"]
    .to_numpy(dtype=bool)
)

validation_metrics["TRACK_ERROR_KM_ALL"] = float(
    np.mean(track_error_km)
)

validation_metrics["TRACK_ERROR_KM_BAY"] = float(
    np.mean(track_error_km[bay_mask])
)

validation_metrics["TRACK_ERROR_KM_ALL_MEDIAN"] = float(
    np.median(track_error_km)
)

validation_metrics["TRACK_ERROR_KM_BAY_MEDIAN"] = float(
    np.median(track_error_km[bay_mask])
)

print("\nVALIDATION METRICS AFTER ONE EPOCH")
print("=" * 70)

for metric, value in validation_metrics.items():
    print(f"{metric:<42} {value:>12.4f}")

# -----------------------------
# Save acceptance artifacts
# -----------------------------

CHECKPOINT_PATH = ROOT / "smoke_model_checkpoint.pt"
METRICS_PATH = ROOT / "smoke_test_metrics.json"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "input_size": len(MODEL_FEATURE_COLUMNS),
        "output_size": len(TARGET_COLUMNS),
        "model_feature_columns": MODEL_FEATURE_COLUMNS,
        "target_columns": TARGET_COLUMNS,
        "seed": SEED,
        "training_epochs": 1,
        "training_loss": training_loss,
        "validation_metrics": validation_metrics,
    },
    CHECKPOINT_PATH,
)

with METRICS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "status": "passed",
            "device": str(device),
            "seed": SEED,
            "training_epochs": 1,
            "training_loss": training_loss,
            "validation_metrics": validation_metrics,
            "checkpoint": str(CHECKPOINT_PATH),
        },
        file,
        indent=2,
    )

assert CHECKPOINT_PATH.exists()
assert CHECKPOINT_PATH.stat().st_size > 0
assert METRICS_PATH.exists()

print("\nCheckpoint:", CHECKPOINT_PATH)
print("Metrics:", METRICS_PATH)
print("\nONE-EPOCH COLAB SMOKE TEST PASSED")

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/bob_cyclone_training_v1")

validation = validation_df.copy()
bay_mask = (
    validation["LAT"].between(5, 25) &
    validation["LON"].between(78, 100)
).to_numpy()

def haversine_km(
    true_lat,
    true_lon,
    predicted_lat,
    predicted_lon,
):
    earth_radius_km = 6371.0088

    true_lat = np.radians(
        np.asarray(true_lat, dtype=float)
    )
    true_lon = np.radians(
        np.asarray(true_lon, dtype=float)
    )
    predicted_lat = np.radians(
        np.asarray(predicted_lat, dtype=float)
    )
    predicted_lon = np.radians(
        np.asarray(predicted_lon, dtype=float)
    )

    delta_lat = predicted_lat - true_lat
    delta_lon = predicted_lon - true_lon

    a = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(true_lat)
        * np.cos(predicted_lat)
        * np.sin(delta_lon / 2.0) ** 2
    )

    return (
        2.0
        * earth_radius_km
        * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )

def safe_mae(actual, predicted, subset_mask=None):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid = (
        np.isfinite(actual) &
        np.isfinite(predicted)
    )

    if subset_mask is not None:
        valid &= np.asarray(subset_mask, dtype=bool)

    if not valid.any():
        return np.nan, 0

    error = np.abs(
        actual[valid] - predicted[valid]
    )

    return float(error.mean()), int(valid.sum())

def track_metrics(
    actual_lat,
    actual_lon,
    predicted_lat,
    predicted_lon,
    subset_mask=None,
):
    actual_lat = np.asarray(actual_lat, dtype=float)
    actual_lon = np.asarray(actual_lon, dtype=float)
    predicted_lat = np.asarray(predicted_lat, dtype=float)
    predicted_lon = np.asarray(predicted_lon, dtype=float)

    valid = (
        np.isfinite(actual_lat) &
        np.isfinite(actual_lon) &
        np.isfinite(predicted_lat) &
        np.isfinite(predicted_lon)
    )

    if subset_mask is not None:
        valid &= np.asarray(subset_mask, dtype=bool)

    errors = haversine_km(
        actual_lat[valid],
        actual_lon[valid],
        predicted_lat[valid],
        predicted_lon[valid],
    )

    return {
        "mean_km": float(errors.mean()),
        "median_km": float(np.median(errors)),
        "p90_km": float(np.percentile(errors, 90)),
        "samples": int(valid.sum()),
    }

# ---------------------------------------------------
# Baseline 1: persistence
# Next state is assumed to equal the current state.
# ---------------------------------------------------

persistence = {
    "lat": validation["LAT"].to_numpy(float),
    "lon": validation["LON"].to_numpy(float),
    "wind": validation["WIND_KTS"].to_numpy(float),
    "pressure": validation["PRESSURE_HPA"].to_numpy(float),
}

# ---------------------------------------------------
# Baseline 2: constant velocity / constant tendency
# Missing previous changes fall back to persistence.
# ---------------------------------------------------

lat_change = (
    validation["LAT_CHANGE_6H"]
    .fillna(0.0)
    .to_numpy(float)
)

lon_change = (
    validation["LON_CHANGE_6H"]
    .fillna(0.0)
    .to_numpy(float)
)

wind_change = (
    validation["WIND_CHANGE_6H_KTS"]
    .fillna(0.0)
    .to_numpy(float)
)

pressure_change = (
    validation["PRESSURE_CHANGE_6H_HPA"]
    .fillna(0.0)
    .to_numpy(float)
)

constant_velocity = {
    "lat": validation["LAT"].to_numpy(float) + lat_change,
    "lon": validation["LON"].to_numpy(float) + lon_change,
    "wind": validation["WIND_KTS"].to_numpy(float) + wind_change,
    "pressure": (
        validation["PRESSURE_HPA"].to_numpy(float)
        + pressure_change
    ),
}

actual = {
    "lat": validation["TARGET_LAT_6H"].to_numpy(float),
    "lon": validation["TARGET_LON_6H"].to_numpy(float),
    "wind": validation[
        "TARGET_WIND_KTS_6H"
    ].to_numpy(float),
    "pressure": validation[
        "TARGET_PRESSURE_HPA_6H"
    ].to_numpy(float),
}

baseline_predictions = {
    "persistence": persistence,
    "constant_velocity": constant_velocity,
}

baseline_rows = []
baseline_details = {}

for baseline_name, predictions in baseline_predictions.items():
    all_track = track_metrics(
        actual["lat"],
        actual["lon"],
        predictions["lat"],
        predictions["lon"],
    )

    bay_track = track_metrics(
        actual["lat"],
        actual["lon"],
        predictions["lat"],
        predictions["lon"],
        subset_mask=bay_mask,
    )

    wind_mae, wind_samples = safe_mae(
        actual["wind"],
        predictions["wind"],
    )

    pressure_mae, pressure_samples = safe_mae(
        actual["pressure"],
        predictions["pressure"],
    )

    bay_wind_mae, bay_wind_samples = safe_mae(
        actual["wind"],
        predictions["wind"],
        subset_mask=bay_mask,
    )

    bay_pressure_mae, bay_pressure_samples = safe_mae(
        actual["pressure"],
        predictions["pressure"],
        subset_mask=bay_mask,
    )

    baseline_details[baseline_name] = {
        "track_all": all_track,
        "track_bay": bay_track,
        "wind_mae_kts": wind_mae,
        "wind_samples": wind_samples,
        "pressure_mae_hpa": pressure_mae,
        "pressure_samples": pressure_samples,
        "bay_wind_mae_kts": bay_wind_mae,
        "bay_wind_samples": bay_wind_samples,
        "bay_pressure_mae_hpa": bay_pressure_mae,
        "bay_pressure_samples": bay_pressure_samples,
    }

    baseline_rows.append({
        "model": baseline_name,
        "track_mean_km_all": all_track["mean_km"],
        "track_median_km_all": all_track["median_km"],
        "track_p90_km_all": all_track["p90_km"],
        "track_mean_km_bay": bay_track["mean_km"],
        "track_median_km_bay": bay_track["median_km"],
        "track_p90_km_bay": bay_track["p90_km"],
        "wind_mae_kts": wind_mae,
        "pressure_mae_hpa": pressure_mae,
        "bay_wind_mae_kts": bay_wind_mae,
        "bay_pressure_mae_hpa": bay_pressure_mae,
    })

baseline_table = pd.DataFrame(baseline_rows)

display(
    baseline_table.round(3)
)

BASELINE_CSV = ROOT / "validation_baselines.csv"
BASELINE_JSON = ROOT / "validation_baselines.json"

baseline_table.to_csv(
    BASELINE_CSV,
    index=False,
)

with BASELINE_JSON.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        baseline_details,
        file,
        indent=2,
    )

print("\nValidation rows:", len(validation))
print("Bay validation rows:", int(bay_mask.sum()))
print("Saved:", BASELINE_CSV)
print("Saved:", BASELINE_JSON)
print("\nVALIDATION BASELINES COMPLETED")

In [ ]:
from pathlib import Path
from google.colab import files
import json
import shutil

ROOT = Path("/content/bob_cyclone_training_v1")
EXPORT_DIR = Path("/content/v1_acceptance_results")

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

artifacts = [
    ROOT / "training_manifest.json",
    ROOT / "normalization_train_only.json",
    ROOT / "colab_preprocessing_contract.json",
    ROOT / "smoke_model_checkpoint.pt",
    ROOT / "smoke_test_metrics.json",
    ROOT / "validation_baselines.csv",
    ROOT / "validation_baselines.json",
]

for artifact in artifacts:
    assert artifact.exists(), (
        f"Required artifact is missing: {artifact}"
    )

    shutil.copy2(
        artifact,
        EXPORT_DIR / artifact.name,
    )

acceptance_summary = {
    "dataset": "bob_cyclone_training_v1",
    "dataset_zip_sha256": (
        "469a6da0b9cb76c4e794feded1cf511799"
        "bd9ed7593436b0845b373772e6b0cb"
    ),
    "status": "accepted",
    "train_samples": 4560,
    "validation_samples": 854,
    "test_samples": 1018,
    "train_cyclones": 235,
    "validation_cyclones": 50,
    "test_cyclones": 65,
    "model_inputs": 46,
    "original_features": 39,
    "missingness_masks": 7,
    "primary_track_baseline": {
        "model": "constant_velocity",
        "validation_mean_error_all_km": 39.040,
        "validation_mean_error_bay_km": 37.932,
    },
    "primary_wind_baseline": {
        "model": "persistence",
        "validation_mae_kts": 2.431,
        "bay_validation_mae_kts": 2.353,
    },
    "primary_pressure_baseline": {
        "model": "persistence",
        "validation_mae_hpa": 1.494,
        "bay_validation_mae_hpa": 1.447,
    },
    "test_partition_used_for_final_metrics": False,
    "next_stage": (
        "Build North Indian Ocean v2 pretraining dataset "
        "while preserving v1 validation and test partitions"
    ),
}

with (
    EXPORT_DIR / "v1_acceptance_summary.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        acceptance_summary,
        file,
        indent=2,
    )

archive_path = shutil.make_archive(
    "/content/bob_cyclone_v1_acceptance_results",
    "zip",
    EXPORT_DIR,
)

print("Acceptance archive created:", archive_path)
print("Files included:")

for path in sorted(EXPORT_DIR.iterdir()):
    print(" ", path.name)

files.download(archive_path)

In [ ]:
from notebookutils import mssparkutils

print("LAKEHOUSE TABLES")
print("=" * 80)

spark.sql("SHOW TABLES").show(
    200,
    truncate=False,
)

print("\nPOSSIBLE IBTrACS SOURCE FILES")
print("=" * 80)

search_terms = (
    "ibtracs",
    "ib_tracs",
    "cyclone",
    "track",
)

matches = []

def scan_folder(path, depth=0, max_depth=5):
    if depth > max_depth:
        return

    try:
        items = mssparkutils.fs.ls(path)
    except Exception:
        return

    for item in items:
        item_path = item.path
        item_name = item.name.lower()

        if any(term in item_name for term in search_terms):
            matches.append({
                "name": item.name,
                "path": item_path,
                "size_mb": round(
                    getattr(item, "size", 0) / (1024**2),
                    3,
                ),
                "is_directory": getattr(
                    item,
                    "isDir",
                    False,
                ),
            })

        if getattr(item, "isDir", False):
            scan_folder(
                item_path,
                depth=depth + 1,
                max_depth=max_depth,
            )

scan_folder("Files")

if matches:
    for match in matches:
        print(match)
else:
    print("No matching source files found under Files.")

print("\nCURRENT V1 TABLE COUNTS")
print("=" * 80)

v1_tables = [
    "dbo.silver_ibtracs_bob_tracks",
    "dbo.silver_cyclone_event_index",
    "dbo.gold_cyclone_feature_master",
    "dbo.gold_track_intensity_6h",
]

for table_name in v1_tables:
    try:
        count = spark.table(table_name).count()
        print(f"{table_name:<42} {count:>8,}")
    except Exception as error:
        print(
            f"{table_name:<42} "
            f"NOT FOUND: {str(error)[:100]}"
        )

print("\nV2 inventory completed.")